# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os

import json
from dotenv import load_dotenv
from lib.agents import Agent
from lib.llm import LLM
from lib.vector_db import VectorStoreManager
from tavily import TavilyClient
from pydantic import BaseModel, Field
from typing import List, Dict
from datetime import datetime
from lib.memory import LongTermMemory, MemoryFragment
from lib.tooling import tool, Tool

E0000 00:00:1780290734.166749 7392385 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1780290734.166772 7392385 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1780290734.166773 7392385 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1780290734.166775 7392385 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1780290734.166776 7392385 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.


In [3]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CHROMA_OPENAI_API_KEY = os.getenv("CHROMA_OPENAI_API_KEY") or OPENAI_API_KEY
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

assert OPENAI_API_KEY is not None, "OPENAI_API_KEY missing in .env"
assert CHROMA_OPENAI_API_KEY is not None, "CHROMA_OPENAI_API_KEY missing in .env"
assert TAVILY_API_KEY is not None, "TAVILY_API_KEY missing in .env"

In [4]:
# Re-open the persistent vector store created in Part 1, via the same reusable manager.
manager = VectorStoreManager(
    openai_api_key=CHROMA_OPENAI_API_KEY,
    persist_path="chromadb",
    embedding_model="text-embedding-3-small"
)

store = manager.get_store("udaplay")

assert store is not None, "Run Udaplay_01_solution_project.ipynb first to build the 'udaplay' store."

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [5]:
@tool
def retrieve_game(query: str) -> List[Dict]:
    """Semantic search: Finds most relevant results in the vector DB.

    args:
    - query: a question about the game industry.

    You'll receive results as a list. Each element contains:
    - Platform: like Game Boy, PlayStation 5, Xbox 360, ...
    - Name: Name of the game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    results = store.query(query_texts=[query], n_results=3)
    docs = []
    for meta, distance in zip(results["metadatas"][0], results["distances"][0]):
        docs.append({
            "Platform": meta.get("Platform"),
            "Name": meta.get("Name"),
            "YearOfRelease": meta.get("YearOfRelease"),
            "Genre": meta.get("Genre"),
            "Publisher": meta.get("Publisher"),
            "Description": meta.get("Description"),
            "distance": distance
        })
    return docs

#### Evaluate Retrieval Tool

In [6]:
class EvaluationReport(BaseModel):
    """Result returned by the evaluate_retrieval tool."""
    useful: bool = Field(
        description="Whether the retrieved documents are sufficient to answer the question"
    )
    description: str = Field(
        description="Detailed explanation of the evaluation result"
    )

In [7]:
# llm as judge
judge_llm = LLM(model="gpt-4o-mini", temperature=0.0) #temperature 0

@tool
def evaluate_retrieval(question: str, retrieved_docs: List[Dict]) -> Dict:
    """Based on the user's question and on the list of retrieved documents,
    analyze the usability of the documents to respond to that question.

    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database

    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    formatted_docs = json.dumps(retrieved_docs, indent=2, default=str)
    prompt = (
    "Your task is to evaluate whether the retrieved documents contain enough "
    "information to answer the user's question.\n\n"
    "Decision rules:\n"
    "- Set useful=True if at least one retrieved document directly answers the question.\n"
    "- Do not require all retrieved documents to be relevant.\n"
    "- Ignore unrelated documents if one strong document is sufficient.\n"
    "- Set useful=False only if none of the documents directly answer the question, "
    "or if the answer would require current/web information.\n"
    "- Prefer the highest-ranked documents because they are ordered by semantic similarity.\n\n"
    f"# Question:\n{question}\n\n"
    f"# Retrieved documents:\n{formatted_docs}\n"
)
    ai_message = judge_llm.invoke(input=prompt, response_format=EvaluationReport)
    report = EvaluationReport.model_validate_json(ai_message.content)
    return report.model_dump()

#### Game Web Search Tool

In [8]:
# client instance for Tavily API
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

@tool
def game_web_search(question: str) -> Dict:
    """Web search fallback: looks up information about the game industry on the web.

    args:
    - question: a question about the game industry.
    """
    search_result = tavily_client.search(
        query=question,
        search_depth="advanced",
        include_answer=True,
        include_raw_content=False,
        include_images=False,
    )
    return {
        "answer": search_result.get("answer", ""),
        "results": [
            {k: r.get(k) for k in ("title", "url", "content")}
            for r in search_result.get("results", [])
],
        "search_metadata": {
            "timestamp": datetime.now().isoformat(),
            "query": question
        }
    }

#### Long Term Memory

In [9]:
# Long-term memory tool - register
def build_memory_registration_tool(ltm: LongTermMemory, owner: str, namespace: str):
    def _register(content: str):
        ltm.register(MemoryFragment(content=content, owner=owner, namespace=namespace))
        return "Saved new memory"
    return Tool(
        func=_register,
        name="register_memory",
        description="Save a fact for later.\nArgs:\n    content: the fact to save",
    )

In [10]:
# Long-term memory tool - search
def build_memory_search_tool(ltm: LongTermMemory, owner: str, namespace: str):
    def _search(content: str):
        result = ltm.search(query_text=content, owner=owner, namespace=namespace, limit=3)
        return str(tuple(zip(result.fragments, result.metadata["distances"])))
    return Tool(
        func=_search,
        name="search_memory",
        description="Look up a saved fact.\nArgs:\n    content: what to search for",
    )

### Agent

In [11]:
AGENT_INSTRUCTIONS = (
    "You are UdaPlay, a research agent for video games.\n\n"
    "Steps for every question:\n"
    "1. First call `search_memory`. If it already has the answer, use it and stop.\n"
    "2. Else search the internal database with `retrieve_game`.\n"
    "3. Check the results with `evaluate_retrieval`.\n"
    "4. Results useful? Answer from them and name the game, platform and year.\n"
    "5. Not useful? Use `game_web_search` and answer from the web, with the links.\n\n"
    "If the user asks you to remember a fact, use `register_memory`.\n\n"
    "Write the answer like this:\n\n"
    "User question:\n<the question>\n\n"
    "Answer:\n<your answer>\n\n"
    "Sources:\n"
    "- Local database (game, platform, year), or\n"
    "- Web links like [Title](https://...), or\n"
    "- Memory (if the answer came from `search_memory`)\n\n"
    "Tools used:\n"
    "- one bullet per tool, skip this if you used none\n\n"
    "Confidence: high, medium or low\n"
)

# Long Term Memory
ltm = LongTermMemory(db=manager)
MEMORY_OWNER, MEMORY_NAMESPACE = "udaplay", "games"

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=AGENT_INSTRUCTIONS,
    tools=[
        build_memory_search_tool(ltm, MEMORY_OWNER, MEMORY_NAMESPACE),
        retrieve_game,
        evaluate_retrieval,
        game_web_search,
        build_memory_registration_tool(ltm, MEMORY_OWNER, MEMORY_NAMESPACE)
    ],
    temperature=0.0
)

In [12]:
# print all messages of a run (from module 06/08)
def print_messages(messages):
    for m in messages:
        print(f" -> role={m.role}, content={m.content}, tool_calls={getattr(m, 'tool_calls', None)}")

# which tools were called
def tools_used(run):
    messages = run.get_final_state()["messages"]
    names = []
    for m in messages:
        if getattr(m, "tool_calls", None):
            for tc in m.tool_calls:
                names.append(tc.function.name)
    return names

# last assistant answer of a run
def final_answer(run):
    for m in reversed(run.get_final_state()["messages"]):
        if m.role == "assistant" and m.content:
            return m.content
    return ""

# print a run nicely
def show_run(title, run):
    print("=" * 80)
    print(title)
    messages = run.get_final_state()["messages"]
    print_messages(messages)
    print("Tools used:", tools_used(run))
    print("Answer:", final_answer(run))

In [13]:
q1 = "When was Pokémon Gold and Silver released?"
run_1 = agent.invoke(query=q1, session_id="pokemon")
show_run("Q1: " + q1, run_1)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Q1: When was Pokémon Gold and Silver released?
 -> role=system, content=You are UdaPlay, a research agent for video games.

Steps for every question:
1. First call `search_memory`. If it already has the answer, use it and stop.
2. Else search the internal database with `retrieve_game`.
3. Check the results with `evaluate_retrieval`.
4. Results useful? Answer from them and name the game, platform and year.
5. Not useful? Use `game_web_search` and answer from the web, with the links.

If the user asks you to remember a fact, use `register_memory`.

Write the

In [14]:
q2 = "Which one was the first 3D platformer Mario game?"
run_2 = agent.invoke(query=q2, session_id="mario")
show_run("Q2: " + q2, run_2)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Q2: Which one was the first 3D platformer Mario game?
 -> role=system, content=You are UdaPlay, a research agent for video games.

Steps for every question:
1. First call `search_memory`. If it already has the answer, use it and stop.
2. Else search the internal database with `retrieve_game`.
3. Check the results with `evaluate_retrieval`.
4. Results useful? Answer from them and name the game, platform and year.
5. Not useful? Use `game_web_search` and answer from the web, with the links.

If the user asks you to remember a fact, use `register_memory`.

Wr

In [15]:
q3 = "Was Mortal Kombat X released for PlayStation 5?"
run_3 = agent.invoke(query=q3, session_id="mkx")
show_run("Q3: " + q3, run_3)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Q3: Was Mortal Kombat X released for PlayStation 5?
 -> role=system, content=You are UdaPlay, a research agent for video games.

Steps for every question:
1. First call `search_memory`. If it already has the answer, use it and stop.
2. Else search the internal database with `retrieve_game`.
3. Check the results with `evaluate_retrieval`.
4. Results useful? Answer from them and name the game, platform and year.
5. Not useful? Use `game_web_search` and answer from the web, with the links.

If the user asks you to remember a fact, use `register_memory`.

Writ

In [16]:
# follow-up in the same session -> tests short-term memory
q2b = "And on which console was it released?"
run_2b = agent.invoke(query=q2b, session_id="mario")  # same session as q2
show_run("Q2 follow-up: " + q2b, run_2b)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Q2 follow-up: And on which console was it released?
 -> role=system, content=You are UdaPlay, a research agent for video games.

Steps for every question:
1. First call `search_memory`. If it already has the answer, use it and stop.
2. Else search the internal database with `retrieve_game`.
3. Check the results with `evaluate_retrieval`.
4. Results useful? Answer from them and name the game, platform and year.
5. Not useful? Use `game_web_search` and answer from the web, with the links.

If the user asks you to remember a fact, use `register_memory`.

Writ

### (Advanced) Long-term memory

Two extra tools:
- `search_memory` checked first.
- `register_memory` saves a fact when asked.

Proof:
1. Ask something not in the local DB. Agent answers from the web. A second turn tells it to remember the fact.
2. Check the memory store directly.
3. Ask the same thing in a new session. No web search. It came from memory.

In [17]:
# step 1 (learn): answer the question -> needs a web search (not in local DB)
q_learn = "Who developed Fifa?"
run_learn = agent.invoke(query=q_learn, session_id="ltm_learn")
show_run("Learn: " + q_learn, run_learn)

# step 2 (save): own turn -> register_memory is the only job, so it fires reliably
answer = final_answer(run_learn)
agent.invoke(query=f"Remember this for next time: {q_learn} {answer}", session_id="ltm_learn")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Learn: Who developed Fifa?
 -> role=system, content=You are UdaPlay, a research agent for video games.

Steps for every question:
1. First call `search_memory`. If it already has the answer, use it and stop.
2. Else search the internal database with `retrieve_game`.
3. Check the results with `evaluate_retrieval`.
4. Results useful? Answer from them and name the game, platform and year.
5. Not useful? Use `game_web_search` and answer from the web, with the links.

If the user asks you to remember a fact, use `register_memory`.

Write the answer like this:



Run('29a5e7ac-93c5-4b3d-a9e8-050dbfc62e7a')

In [18]:
# check the memory store directly -> the fact should be there now
res = ltm.search(query_text=q_learn, owner=MEMORY_OWNER, namespace=MEMORY_NAMESPACE, limit=3)
for f in res.fragments:
    print("stored:", f.content)
assert res.fragments, "nothing was saved to memory"

stored: FIFA video games were originally developed by EA Sports. However, since 2023, they are being developed by Delphi Interactive as part of a new partnership with Netflix.


In [19]:
# same question, new session -> no web search means it came from memory
run_recall = agent.invoke(query=q_learn, session_id="ltm_recall")
show_run("Recall: " + q_learn, run_recall)

used = tools_used(run_recall)
assert "game_web_search" not in used, f"should come from memory, but web ran: {used}"
print("OK - answer came from long-term memory, no web search")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Recall: Who developed Fifa?
 -> role=system, content=You are UdaPlay, a research agent for video games.

Steps for every question:
1. First call `search_memory`. If it already has the answer, use it and stop.
2. Else search the internal database with `retrieve_game`.
3. Check the results with `evaluate_retrieval`.
4. Results useful? Answer from them and name the game, platform and year.
5. Not useful? Use `game_web_search` and answer from the web, with the links.

If the user asks you to remember a fact, use `register_memory`.

Write the answer like this:

User question:
<the question>

Answer:
<your answer>

Sources:
- Local database (game, platform, year), or
- Web links like [Title](https://...), or
- Memory (if the answer came f